# Capítulo 7 — Classes e herança: os sensores do rover

## Missão
Modelar as peças do rover como **objetos**: uma classe base `Sensor` e sensores especializados por **herança** (temperatura, distância, bateria), com atributos, métodos, propriedades e representação em texto.

**No fim deste caderno você terá:** o módulo `gabarito_rover/sensores.py`, com a hierarquia de sensores que os próximos capítulos usam.

**Leitura essencial (opcional, livro em inglês):**
- Creating a Class · PDF p. 238
- Giving an Object Its Attributes · PDF p. 240
- Giving a Class Methods · PDF p. 246
- Understanding Class Inheritance · PDF p. 256

**Gabarito:** a mesma trilha com o código pronto. Tente primeiro em `praticar/`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados da missão do rover)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote gabarito_rover/
os.makedirs("../gabarito_rover", exist_ok=True)
open("../gabarito_rover/__init__.py", "a").close()        # gabarito_rover/ é o rover que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: gabarito_rover")

## Ideia em 30 segundos
Uma **classe** é a planta, e o **objeto** é a peça construída a partir dela.
- `__init__` prepara o objeto, e `self` é o próprio objeto.
- **Herança** (`class Filho(Pai)`) reaproveita o que é comum e muda só o que é diferente; `super()` chama o pai.
- `__repr__` diz como o objeto aparece quando é mostrado.

### 7.1 **Primeira classe.** Escreva `Motor` com `nome` e `velocidade` (começa em 0), os métodos `acelerar(delta)` (sem passar de 100) e `parar()`, e `__repr__`. Crie dois motores e mostre que cada um tem o seu estado.

In [ ]:
class Motor:
    def __init__(self, nome):
        self.nome = nome
        self.velocidade = 0

    def acelerar(self, delta):
        self.velocidade = min(100, self.velocidade + delta)

    def parar(self):
        self.velocidade = 0

    def __repr__(self):
        return f"Motor({self.nome!r}, velocidade={self.velocidade})"

esq, dir_ = Motor("esquerdo"), Motor("direito")
esq.acelerar(70); esq.acelerar(50); dir_.acelerar(30)
print(esq, dir_)

### 7.2 **Atributo de classe × de instância.** Acrescente a `Motor` um atributo de classe `total = 0` que conta quantos motores foram criados (incremente no `__init__` com `Motor.total += 1`). Crie 3 e mostre a contagem.

In [ ]:
class Motor:
    total = 0

    def __init__(self, nome):
        self.nome = nome
        self.velocidade = 0
        Motor.total += 1

for n in ("A", "B", "C"):
    Motor(n)
print(Motor.total)

### 7.3 **Propriedade com validação.** Escreva a classe `Bateria` com uma `@property` `nivel` cujo *setter* recusa valores fora de 0–100 (`ValueError`). Teste um valor válido e um inválido.

In [ ]:
class Bateria:
    def __init__(self, nivel=100):
        self.nivel = nivel

    @property
    def nivel(self):
        return self._nivel

    @nivel.setter
    def nivel(self, valor):
        if not 0 <= valor <= 100:
            raise ValueError(f"nível inválido: {valor}")
        self._nivel = valor

b = Bateria(); b.nivel = 55; print(b.nivel)
try:
    b.nivel = 140
except ValueError as e:
    print(e)

### 7.4 **Hierarquia de sensores.** Escreva `sensores.py` com:
- `Sensor(nome, unidade)`: guarda um histórico; `registrar(valor)` valida com `validar()` (no pai, aceita tudo) e devolve `False` se recusar; `ultimo` (propriedade); `media()`; `__repr__`;
- `SensorTemperatura`: unidade `°C`, válido entre -130 e 40;
- `SensorDistancia`: unidade `cm`, válido entre 0 e 400, com o método `obstaculo(limite=50)`;
- `SensorBateria`: unidade `%`, válido entre 0 e 100, com o método `critica(minima=30)`.

Os filhos usam `super().__init__`.

_A célula grava `gabarito_rover/sensores.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../gabarito_rover/sensores.py
"""Sensores do rover: uma classe base e especializações por herança."""


class Sensor:
    def __init__(self, nome: str, unidade: str):
        self.nome = nome
        self.unidade = unidade
        self.historico: list[float] = []

    def validar(self, valor: float) -> bool:
        return True

    def registrar(self, valor: float) -> bool:
        if not self.validar(valor):
            return False
        self.historico.append(valor)
        return True

    @property
    def ultimo(self):
        return self.historico[-1] if self.historico else None

    def media(self) -> float:
        return sum(self.historico) / len(self.historico) if self.historico else 0.0

    def __repr__(self):
        return f"{type(self).__name__}({self.nome!r}, último={self.ultimo} {self.unidade}, n={len(self.historico)})"


class SensorTemperatura(Sensor):
    def __init__(self, nome: str = "temperatura"):
        super().__init__(nome, "°C")

    def validar(self, valor: float) -> bool:
        return -130 <= valor <= 40


class SensorDistancia(Sensor):
    def __init__(self, nome: str = "distancia"):
        super().__init__(nome, "cm")

    def validar(self, valor: float) -> bool:
        return 0 <= valor <= 400

    def obstaculo(self, limite: float = 50) -> bool:
        return self.ultimo is not None and self.ultimo < limite


class SensorBateria(Sensor):
    def __init__(self, nome: str = "bateria"):
        super().__init__(nome, "%")

    def validar(self, valor: float) -> bool:
        return 0 <= valor <= 100

    def critica(self, minima: float = 30) -> bool:
        return self.ultimo is not None and self.ultimo < minima


### 7.5 **Teste os sensores.** Crie um de cada, registre os valores de `LEITURAS` (mais um valor inválido de temperatura, 99) e mostre os objetos, a média da temperatura e se há obstáculo na última leitura.

In [ ]:
from gabarito_rover.sensores import SensorTemperatura, SensorDistancia, SensorBateria
t, d, b = SensorTemperatura(), SensorDistancia(), SensorBateria()
for hora, temp, poeira, bat, dist in LEITURAS:
    t.registrar(temp); d.registrar(dist); b.registrar(bat)
print("aceitou 99 °C?", t.registrar(99))
print(t, d, b, sep="\n")
print(round(t.media(), 1), d.obstaculo(), b.critica())

### 7.6 **Polimorfismo.** Coloque os 3 sensores numa lista e, num único `for`, mostre `nome`, `ultimo` e `unidade`. Use `isinstance` para chamar `obstaculo()` só no sensor de distância.

In [ ]:
for s in (t, d, b):
    extra = f" | obstáculo: {s.obstaculo()}" if isinstance(s, SensorDistancia) else ""
    print(f"{s.nome:12} {s.ultimo} {s.unidade}{extra}")

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
